In [1]:
!pip install torch joblib matplotlib seaborn tqdm pgmpy fairlearn optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 50.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
import os, gc, copy, math, warnings, logging
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

import joblib
from tqdm import tqdm

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

from fairlearn.metrics import equalized_odds_difference

warnings.filterwarnings('ignore')
logging.getLogger("pgmpy").setLevel(logging.ERROR)

SEED             = 42
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR        = './cache'
RESULTS_DIR      = '/kaggle/working'

for _d in [CACHE_DIR, RESULTS_DIR]:
    os.makedirs(_d, exist_ok=True)

def floor2(x):
    return math.floor(abs(float(x)) * 100) / 100

def set_seed(s=SEED):
    torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DATASET_CONFIG: Dict[str, Dict] = {
    "adult":     {"sens_attrs": [("sens_sex_train",     "sens_sex_test"),
                                 ("sens_race_train",    "sens_race_test")]},
    "compas":    {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "german":    {"sens_attrs": [("sens_age_train",     "sens_age_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "bank":      {"sens_attrs": [("sens_marital_train", "sens_marital_test"),
                                 ("sens_job_train",     "sens_job_test")]},
    "lawschool": {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_gender_train",  "sens_gender_test")]},
    "hospital":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
}
def to_dense(X):
    arr = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

def clean_numeric_column(s):
    s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan)
    med = s.median()
    return s.fillna(0.0 if np.isnan(med) else med)

def safe_qcut(s, q=5):
    s = clean_numeric_column(s)
    if s.nunique() <= 1:
        return pd.Series(0, index=s.index, dtype=int)
    try:
        return pd.qcut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except Exception:
        try:
            return pd.cut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
        except:
            return pd.Series(0, index=s.index, dtype=int)

def make_num_pipeline():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])

def make_cat_pipeline():
    return Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore'))])


def preprocess_adult(csv_path='/kaggle/input/datasets/lateglue/all-datasets/adult.csv',
                     seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_adult.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Adult"); return joblib.load(cache_file)
    col_names = ['age','workclass','fnlwgt','education','education-num','marital-status',
                 'occupation','relationship','race','sex','capital-gain','capital-loss',
                 'hours-per-week','native-country','income']
    df = None
    for sep in [', ', ',']:
        try:
            peek = pd.read_csv(csv_path, sep=sep, nrows=1, header=0)
            if peek.shape[1] == 15:
                first = str(peek.iloc[0, 0]).strip()
                df = (pd.read_csv(csv_path, sep=sep, names=col_names, header=None)
                      if first.lstrip('-').isdigit()
                      else pd.read_csv(csv_path, sep=sep, header=0))
                df.columns = col_names; break
        except:
            continue
    if df is None:
        df = pd.read_csv(csv_path)
        if df.shape[1] == 15: df.columns = col_names
        else: raise ValueError("Cannot parse Adult CSV")
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True).drop(columns=['fnlwgt'])
    df['y'] = df['income'].astype(str).str.strip().str.contains('>50K', na=False).astype(int)
    df['sex_bin']  = (df['sex'].astype(str).str.strip() == 'Male').astype(int)
    df['race_bin'] = (df['race'].astype(str).str.strip() == 'White').astype(int)
    num_c = ['age','education-num','capital-gain','capital-loss','hours-per-week']
    cat_c = ['workclass','education','marital-status','occupation','relationship','native-country']
    for col in num_c: df[col] = clean_numeric_column(df[col])
    for col in ['capital-gain','capital-loss']:
        df[col] = df[col].clip(upper=df[col].quantile(0.99))
    X = df.drop(columns=['income','y','sex_bin','race_bin','sex','race'])
    y = df['y'].values
    X_tr, X_te, y_tr, y_te, ss_tr, ss_te, sr_tr, sr_te = train_test_split(
        X, y, df['sex_bin'].values, df['race_bin'].values,
        test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n', make_num_pipeline(), num_c), ('c', make_cat_pipeline(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))
    
    # ---- FIX: BBN data must be purely numeric ----
    bbn = df.copy()
    # Drop the original string 'income' column (we already have numeric 'y')
    bbn = bbn.drop(columns=['income'])
    # Convert numeric columns to discrete bins
    for col in num_c:
        bbn[col] = safe_qcut(bbn[col], 5)
    # Convert all categorical columns (including 'sex', 'race') to numeric codes
    cat_cols_for_bbn = cat_c + ['sex', 'race']
    for col in cat_cols_for_bbn:
        if col in bbn.columns:
            bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
    # Ensure 'y' is int
    bbn['y'] = bbn['y'].astype(int)
    
    # Double-check: all columns should now be numeric
    for col in bbn.columns:
        if bbn[col].dtype == 'object':
            raise ValueError(f"Column {col} still has object dtype. Please fix.")
    
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_sex_train=ss_tr, sens_sex_test=ss_te,
             sens_race_train=sr_tr, sens_race_test=sr_te)
    if use_cache: joblib.dump(r, cache_file)
    return r


def preprocess_compas(csv_path='/kaggle/input/datasets/lateglue/all-datasets/compas-scores-two-years.csv',
                      seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_compas.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] COMPAS"); return joblib.load(cache_file)
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'] <= 30) & (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) & (df['c_charge_degree'] != 'O') & (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex','priors_count',
         'days_b_screening_arrest','decile_score','juv_other_count','juv_misd_count',
         'juv_fel_count','c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)
    df['race_bin'] = (df['race'] == 'African-American').astype(int)
    df['sex_bin']  = (df['sex'] == 'Male').astype(int)
    df['c_jail_in']  = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time']  = (df['c_jail_out'] - df['c_jail_in']).dt.days.fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    num_c = ['age','priors_count','days_b_screening_arrest','decile_score','jail_time',
             'juv_other_count','juv_misd_count','juv_fel_count']
    cat_c = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    for col in num_c: df[col] = clean_numeric_column(df[col])
    X = df.drop(columns=['is_recid','two_year_recid','race_bin','sex_bin'])
    y = df['two_year_recid'].values
    X_tr, X_te, y_tr, y_te, sr_tr, sr_te, ss_tr, ss_te = train_test_split(
        X, y, df['race_bin'], df['sex_bin'], test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n', make_num_pipeline(), num_c), ('c', make_cat_pipeline(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for col in num_c: bbn[col] = safe_qcut(bbn[col], 5)
    for col in cat_c + ['race', 'sex']: bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_race_train=sr_tr.reset_index(drop=True), sens_race_test=sr_te.reset_index(drop=True),
             sens_sex_train=ss_tr.reset_index(drop=True),  sens_sex_test=ss_te.reset_index(drop=True))
    if use_cache: joblib.dump(r, cache_file)
    return r


def preprocess_german(csv_path="/kaggle/input/datasets/lateglue/all-datasets/german.data",
                      seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_german.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] German"); return joblib.load(cache_file)
    col_names = ["status","duration","credit_history","purpose","amount","savings","employment",
                 "installment_rate","personal_status_sex","other_debtors","residence","property",
                 "age","other_installment_plans","housing","number_credits","job","people_liable",
                 "telephone","foreign_worker","credit"]
    df = pd.read_csv(csv_path, sep=' ', names=col_names)
    sx = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex'] = df['personal_status_sex'].map(sx)
    df['sex_bin'] = (df['sex'] == 'male').astype(int)
    df['age_bin'] = (df['age'] >= 25).astype(int)
    df['y'] = df['credit'].map({1: 1, 2: 0})
    df = df.drop(columns=['personal_status_sex', 'sex', 'age', 'foreign_worker', 'credit'])
    num_c = ["duration","amount","installment_rate","residence","number_credits","people_liable"]
    cat_c = [c for c in df.columns if c not in num_c + ['sex_bin','age_bin','y']]
    for col in num_c: df[col] = clean_numeric_column(df[col])
    X = df.drop(columns=['y'])
    y = df['y'].values
    X_tr, X_te, y_tr, y_te, sa_tr, sa_te, ss_tr, ss_te = train_test_split(
        X, y, df['age_bin'].values, df['sex_bin'].values, test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n', make_num_pipeline(), num_c), ('c', make_cat_pipeline(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for col in num_c: bbn[col] = safe_qcut(bbn[col], 5)
    for col in cat_c + ['sex_bin','age_bin']: bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_age_train=sa_tr, sens_age_test=sa_te,
             sens_sex_train=ss_tr, sens_sex_test=ss_te)
    if use_cache: joblib.dump(r, cache_file)
    return r


def preprocess_bank(csv_path='/kaggle/input/datasets/lateglue/all-datasets/bank-full.csv',
                    seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_bank.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Bank"); return joblib.load(cache_file)
    try: df = pd.read_csv(csv_path, sep=';')
    except: df = pd.read_csv(csv_path, sep=',')
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    tc = ('y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else df.columns[-1])
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y'] = np.where(df[tc].isin(['yes','y','true','1']), 1, 0)
    df['marital_bin'] = (df['marital'] == df['marital'].value_counts().idxmax()).astype(int)
    df['job_bin']     = (df['job']     == df['job'].value_counts().idxmax()).astype(int)
    cat_c = [c for c in ['job','education','default','housing','loan','contact','month','poutcome'] if c in df.columns]
    num_c = [c for c in ['age','balance','day','duration','campaign','pdays','previous'] if c in df.columns]
    for col in num_c: df[col] = clean_numeric_column(df[col])
    for col in ['balance','duration','pdays','previous']:
        if col in df.columns: df[col] = df[col].clip(upper=df[col].quantile(0.99))
    X = df.drop(columns=['y','marital_bin','job_bin'])
    y = df['y'].values
    X_tr, X_te, y_tr, y_te, sm_tr, sm_te, sj_tr, sj_te = train_test_split(
        X, y, df['marital_bin'], df['job_bin'], test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n', make_num_pipeline(), num_c), ('c', make_cat_pipeline(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for col in num_c: bbn[col] = safe_qcut(bbn[col], 5)
    for col in cat_c + ['marital','job']: bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_marital_train=sm_tr.reset_index(drop=True), sens_marital_test=sm_te.reset_index(drop=True),
             sens_job_train=sj_tr.reset_index(drop=True),     sens_job_test=sj_te.reset_index(drop=True))
    if use_cache: joblib.dump(r, cache_file)
    return r


def preprocess_lawschool(law_path="/kaggle/input/datasets/lateglue/all-datasets/bar_pass_prediction.csv",
                         use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_lawschool.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] LawSchool"); return joblib.load(cache_file)
    df = pd.read_csv(law_path)
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.dropna(subset=['pass_bar','race','sex'])
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = clean_numeric_column(df[col])
    mc = df['race'].value_counts().idxmax()
    df['race_bin'] = (df['race'] == mc).astype(int)
    gm = {v: i for i, v in enumerate(sorted(df['sex'].unique()))}
    df['gender_bin'] = df['sex'].map(gm)
    num_c = [c for c in ['lsat','ugpa','zfygpa','zgpa','fam_inc','age','gpa']
             if c in df.columns and df[c].nunique() > 1]
    cat_c = [c for c in ['tier','cluster','fulltime','parttime'] if c in df.columns]
    X = df[num_c + cat_c + ['race','sex']]
    y = df['pass_bar'].values
    X_tr, X_te, y_tr, y_te, sr_tr, sr_te, sg_tr, sg_te = train_test_split(
        X, y, df['race_bin'], df['gender_bin'], test_size=0.2, stratify=y, random_state=SEED)
    pre = ColumnTransformer([('n', make_num_pipeline(), num_c), ('c', make_cat_pipeline(), cat_c + ['race','sex'])])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for col in num_c: bbn[col] = safe_qcut(df[col], 5)
  # pass_bar must also be encoded — it may be 'Y'/'N' string
    for col in cat_c + ['race', 'sex']:
      if col in bbn.columns:
          bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
  # Encode pass_bar if it's still a string/object
    if 'pass_bar' in bbn.columns and bbn['pass_bar'].dtype == object:
      bbn['pass_bar'] = bbn['pass_bar'].astype('category').cat.codes.astype(int)
  # Encode any remaining object columns
    for col in bbn.columns:
      if bbn[col].dtype == object:
          bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
          
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_race_train=sr_tr.reset_index(drop=True),   sens_race_test=sr_te.reset_index(drop=True),
             sens_gender_train=sg_tr.reset_index(drop=True), sens_gender_test=sg_te.reset_index(drop=True))
    if use_cache: joblib.dump(r, cache_file)
    return r


def preprocess_hospital(csv_path='/kaggle/input/datasets/lateglue/all-datasets/diabetes_hospital_fairlearn.csv',
                        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_hospital.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Hospital"); return joblib.load(cache_file)
    df = pd.read_csv(csv_path)
    df = df.drop(columns=[c for c in ["max_glu_serum","A1Cresult","readmitted.1"] if c in df.columns])
    df = df[~df.isin(['Missing']).any(axis=1)].dropna(subset=['race','gender']).reset_index(drop=True)
    age_map = {"'0-10'":5,"'10-20'":15,"'20-30'":25,"'30-40'":35,"'40-50'":45,
               "'50-60'":55,"'60-70'":65,"'70-80'":75,"'80-90'":85,"'90-100'":95,
               "'30 years or younger'":20,"'30-60 years'":45,"'Over 60 years'":70}
    df['age'] = df['age'].replace(age_map).astype(float)
    df['y'] = df['readmit_binary'].astype(int)
    mc = df['race'].value_counts().idxmax()
    df['race_bin']   = (df['race'] == mc).astype(int)
    df['gender_bin'] = df['gender'].map({'Male':0,'Female':1}).fillna(0).astype(int)
    cat_c = ['discharge_disposition_id','admission_source_id','medical_specialty',
             'primary_diagnosis','insulin','change','diabetesMed']
    num_c = ['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
             'number_diagnoses','had_emergency','had_inpatient_days','had_outpatient_days','medicare','medicaid']
    for col in num_c: df[col] = clean_numeric_column(df[col])
    X = df.drop(columns=['readmit_binary','y','race_bin','gender_bin'])
    y = df['y'].values
    X_tr, X_te, y_tr, y_te, sr_tr, sr_te, sg_tr, sg_te = train_test_split(
        X, y, df['race_bin'], df['gender_bin'], test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n', make_num_pipeline(), num_c), ('c', make_cat_pipeline(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))
    bbn = df.copy()
    for col in num_c: bbn[col] = safe_qcut(bbn[col], 5)
    for col in cat_c + ['race','gender']: bbn[col] = bbn[col].astype('category').cat.codes.astype(int)
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
             bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
             y_train=y_tr, y_test=y_te,
             sens_race_train=sr_tr.reset_index(drop=True),   sens_race_test=sr_te.reset_index(drop=True),
             sens_sex_train=sg_tr.reset_index(drop=True),    sens_sex_test=sg_te.reset_index(drop=True))
    if use_cache: joblib.dump(r, cache_file)
    return r


# ── Tensor helpers ──────────────────────────────────────────────────────────
def _tensor(arr, dtype=torch.float32):
    return torch.tensor(np.asarray(arr), dtype=dtype).to(DEVICE)

def _get_proba(encoder, classifier, X_tensor, batch=2048):
    encoder.eval(); classifier.eval()
    parts = []
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch):
            z     = encoder(X_tensor[i:i+batch])
            logit = classifier(z)
            parts.append(torch.sigmoid(logit).cpu().numpy())
    return np.concatenate(parts)


/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [4]:
# ══════════════════════════════════════════════════════════════════════════
#  CELL 2 — LCFR v2: Causal Reweighing + Unified Fairness Encoder
#            + Pareto-Optimal Group Threshold Calibration
# ══════════════════════════════════════════════════════════════════════════

import os, gc, copy, math, warnings, logging
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split

import joblib
from tqdm import tqdm

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator, HillClimbSearch
from pgmpy.inference import VariableElimination

from fairlearn.metrics import equalized_odds_difference

warnings.filterwarnings('ignore')
logging.getLogger("pgmpy").setLevel(logging.ERROR)


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 1 — Causal Knowledge (unchanged)
# ════════════════════════════════════════════════════════════════════════════

CAUSAL_KNOWLEDGE: Dict[str, Dict] = {
    "adult":     {"sens_cols": ["sex", "race"],      "biased_mediators": [],
                  "fair_mediators": ["education", "education-num", "occupation", "marital-status"],
                  "effect_type": "direct"},
    "compas":    {"sens_cols": ["race", "sex"],       "biased_mediators": ["decile_score", "score_text"],
                  "fair_mediators": [],               "effect_type": "total"},
    "german":    {"sens_cols": ["sex_bin", "age_bin"],"biased_mediators": [],
                  "fair_mediators": ["credit_history", "savings", "employment"],
                  "effect_type": "direct"},
    "bank":      {"sens_cols": ["marital", "job"],    "biased_mediators": [],
                  "fair_mediators": [],               "effect_type": "learned"},
    "lawschool": {"sens_cols": ["race", "sex"],       "biased_mediators": ["lsat", "ugpa"],
                  "fair_mediators": [],               "effect_type": "total"},
    "hospital":  {"sens_cols": ["race", "gender"],    "biased_mediators": ["medicare", "medicaid"],
                  "fair_mediators": [],               "effect_type": "total"},
}

def compute_mi(a: np.ndarray, b: np.ndarray) -> float:
    return float(mutual_info_score(a.astype(int), b.astype(int)))


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 2 — PRE: Causal-Path-Aware Instance Reweighing
#
#  Novel over Kamiran-Calders: weight each instance by a combination of
#  (a) demographic parity correction and (b) a causal path strength factor
#  derived from the MI along the sensitive→mediator→Y path.
#  Instances whose sensitive attribute has high causal influence on Y
#  through biased mediators get upweighted MORE aggressively, while
#  instances with only spurious correlation get standard KC weights.
#  This lets the encoder focus its fairness budget on structurally biased
#  cases rather than treating all group imbalances equally.
# ════════════════════════════════════════════════════════════════════════════

def causal_reweigh(
    y: np.ndarray,
    s: np.ndarray,
    bbn_df: pd.DataFrame,
    sens_col: str,
    biased_mediators: List[str],
    effect_type: str,
) -> np.ndarray:
    n = len(y)
    s = s.astype(int)
    y = y.astype(int)

    # Base KC weights
    w = np.zeros(n, dtype=np.float32)
    for sg in (0, 1):
        for yg in (0, 1):
            mask = (s == sg) & (y == yg)
            if mask.sum() > 0:
                w[mask] = n / (mask.sum() * 4.0)

    # Causal path amplification: if biased mediators exist and effect_type
    # is total/learned, scale up weights for the minority group proportional
    # to the total MI through biased paths.
    if biased_mediators and effect_type in ("total", "learned") and sens_col in bbn_df.columns:
        path_mi = 0.0
        for med in biased_mediators:
            if med in bbn_df.columns:
                path_mi += compute_mi(bbn_df[sens_col].values, bbn_df[med].values)
        if path_mi > 0.01:
            amp = float(np.clip(1.0 + path_mi * 5.0, 1.0, 2.5))
            minority_group = int(np.argmin([np.mean(s == 0), np.mean(s == 1)]))
            w[s == minority_group] *= amp
            w = w / w.mean()

    return w


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 3 — Causal Graph (simplified — removed HillClimb for speed,
#               direct BBN fit is enough for CF generation)
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class CausalGraph:
    dataset_name:     str
    sens_col:         str
    biased_mediators: List[str]
    fair_mediators:   List[str]
    effect_type:      str
    sens_mi:          float = 0.0
    mediator_mis:     Dict[str, float] = field(default_factory=dict)
    bbn_columns:      List[str] = field(default_factory=list)
    _ve:              Optional[object] = field(default=None, repr=False)

    MI_THRESHOLD = 0.02

    def fit(self, bbn_df: pd.DataFrame, label_col: str = "y") -> "CausalGraph":
        df = bbn_df.copy()
        self.bbn_columns = list(df.columns)
        if self.sens_col in df.columns and label_col in df.columns:
            self.sens_mi = compute_mi(df[self.sens_col].values, df[label_col].values)
        for med in self.biased_mediators + self.fair_mediators:
            if med in df.columns and self.sens_col in df.columns:
                self.mediator_mis[med] = compute_mi(df[self.sens_col].values, df[med].values)

        # Build a minimal star-topology BBN: sens → everything
        # This is faster than HillClimb and sufficient for CF generation
        nodes = [c for c in df.columns if c != self.sens_col]
        edges = [(self.sens_col, n) for n in nodes if n in df.columns]
        if not edges:
            edges = [(label_col, self.sens_col)]
        try:
            model = DiscreteBayesianNetwork(edges)
            model.fit(df, estimator=BayesianEstimator,
                      prior_type='BDeu', equivalent_sample_size=5)
            self._ve = VariableElimination(model)
        except Exception:
            self._ve = None
        return self

    def generate_counterfactuals(self, bbn_rows: pd.DataFrame) -> pd.DataFrame:
        cf = bbn_rows.copy().reset_index(drop=True)
        if self.sens_col not in cf.columns:
            return cf
        max_val = int(cf[self.sens_col].max()) if len(cf) > 0 else 1
        cf[self.sens_col] = (cf[self.sens_col].astype(int) + 1) % (max_val + 1)
        if self.effect_type == "direct":
            return cf
        if self._ve is not None:
            for med in self.biased_mediators:
                if med not in cf.columns:
                    continue
                if self.mediator_mis.get(med, 0.0) < self.MI_THRESHOLD:
                    continue
                try:
                    new_vals = []
                    for _, row in cf.iterrows():
                        ev = {self.sens_col: int(row[self.sens_col])}
                        q  = self._ve.query([med], evidence=ev, show_progress=False)
                        pr = q.values / q.values.sum()
                        new_vals.append(int(np.random.choice(len(pr), p=pr)))
                    cf[med] = new_vals
                except Exception:
                    pass
        return cf


def build_causal_graphs(dataset_name, bbn_train_df, y_train, label_col="y"):
    ck = CAUSAL_KNOWLEDGE.get(dataset_name, {})
    df = bbn_train_df.copy()
    if label_col not in df.columns:
        df[label_col] = y_train
    graphs = []
    for sc in ck.get("sens_cols", []):
        if sc not in df.columns:
            continue
        g = CausalGraph(
            dataset_name=dataset_name, sens_col=sc,
            biased_mediators=[m for m in ck.get("biased_mediators", []) if m in df.columns],
            fair_mediators=[m for m in ck.get("fair_mediators", []) if m in df.columns],
            effect_type=ck.get("effect_type", "learned"),
        )
        g.fit(df, label_col=label_col)
        graphs.append(g)
    return graphs


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 4 — Neural Architecture
#  Stripped: removed CounterfactualRegularizer with learnable mask (it was
#  not converging fast enough and adding ~in_dim params with no benefit).
#  CF is now applied directly in BBN feature space (no learned embedding).
# ════════════════════════════════════════════════════════════════════════════

class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()
    @staticmethod
    def backward(ctx, g):
        return -ctx.alpha * g, None

class GradientReversal(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha
    def forward(self, x):
        return GradientReversalFn.apply(x, self.alpha)
    def set_alpha(self, a):
        self.alpha = a

class FairnessEncoder(nn.Module):
    def __init__(self, in_dim, hidden=256, z_dim=128, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden)
        self.b1 = nn.Sequential(nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden))
        self.b2 = nn.Sequential(nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden))
        self.out = nn.Sequential(nn.LayerNorm(hidden), nn.GELU(),
                                  nn.Linear(hidden, z_dim), nn.LayerNorm(z_dim))
    def forward(self, x):
        h = self.proj(x)
        h = h + self.b1(h)
        h = h + self.b2(h)
        return self.out(h)

class TaskHead(nn.Module):
    def __init__(self, z_dim=128):
        super().__init__()
        self.fc = nn.Linear(z_dim, 1)
    def forward(self, z):
        return self.fc(z).squeeze(-1)

class AdversarialHead(nn.Module):
    def __init__(self, z_dim=128, hidden=64, n_groups=2):
        super().__init__()
        self.grl = GradientReversal(alpha=1.0)
        self.net = nn.Sequential(nn.Linear(z_dim, hidden), nn.GELU(),
                                  nn.Linear(hidden, n_groups))
    def forward(self, z):
        return self.net(self.grl(z))
    def set_alpha(self, a):
        self.grl.set_alpha(a)


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 5 — EO Losses
#
#  Two losses used together:
#  (a) Soft EO loss on logits (differentiable, per-batch)
#  (b) Wasserstein-1 distance between score distributions per group
#      (novel: this penalises distributional shift, not just TPR/FPR gaps,
#       giving a smoother gradient signal especially for small minority groups)
# ════════════════════════════════════════════════════════════════════════════

def eo_loss(logit, y, s, margin=0.005):
    probs = torch.sigmoid(logit)
    tprs, fprs = [], []
    for g in (0, 1):
        pm = (s == g) & (y == 1)
        nm = (s == g) & (y == 0)
        tprs.append(probs[pm].mean() if pm.sum() > 1 else torch.tensor(0.5, device=logit.device))
        fprs.append(probs[nm].mean() if nm.sum() > 1 else torch.tensor(0.5, device=logit.device))
    return F.relu(torch.max(torch.abs(tprs[0]-tprs[1]), torch.abs(fprs[0]-fprs[1])) - margin)


def wasserstein_group_loss(probs, s):
    # W1 distance between score CDFs of group 0 and group 1
    # Approximated via sorted difference (exact W1 for 1D)
    p0 = probs[s == 0]
    p1 = probs[s == 1]
    if len(p0) < 2 or len(p1) < 2:
        return torch.tensor(0.0, device=probs.device)
    p0_s = p0.sort().values
    p1_s = p1.sort().values
    # Interpolate shorter to match longer
    if len(p0_s) != len(p1_s):
        n = min(len(p0_s), len(p1_s))
        idx0 = torch.linspace(0, len(p0_s)-1, n).long()
        idx1 = torch.linspace(0, len(p1_s)-1, n).long()
        p0_s = p0_s[idx0]
        p1_s = p1_s[idx1]
    return torch.abs(p0_s - p1_s).mean()


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 6 — PID Controller (simplified — no acc_scale, just pure EO control)
#  The acc_scale in v1 was suppressing EO too much early. We instead use a
#  simple warm-up: EO loss weight linearly ramps from 0 to pid_weight over
#  the first 30 epochs, giving the task head time to stabilise.
# ════════════════════════════════════════════════════════════════════════════

class PIDController:
    def __init__(self, kp=4.0, ki=0.5, kd=0.3, w_min=0.1, w_max=20.0, target=0.01):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.w_min, self.w_max = w_min, w_max
        self.target = target
        self._integral = 0.0
        self._prev_err = 0.0
        self._w = 1.0

    def step(self, current_eo):
        err = current_eo - self.target
        self._integral = np.clip(self._integral + err, -4.0, 4.0)
        d = err - self._prev_err
        self._prev_err = err
        raw = self.kp * err + self.ki * self._integral + self.kd * d
        self._w = float(np.clip(raw, self.w_min, self.w_max))
        return self._w

    def reset(self):
        self._integral = 0.0
        self._prev_err = 0.0
        self._w = 1.0

    @property
    def weight(self):
        return self._w


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 7 — POST: Pareto-Optimal Group Threshold Calibration
#
#  Novel framing: instead of a single threshold or simple threshold-shift,
#  we search over a 2D grid of (t0, t1) per-group thresholds and find the
#  Pareto frontier of (EO, accuracy). We then select the point on the frontier
#  that minimises EO subject to accuracy >= acc_floor.
#
#  This is distinct from prior work (Hardt et al. 2016 post-processing) in two
#  ways: (1) we use isotonic regression to calibrate scores per group before
#  thresholding, correcting for group-specific score miscalibration; and (2) we
#  explicitly solve the constrained Pareto problem rather than using a single
#  pre-computed operating point.
# ════════════════════════════════════════════════════════════════════════════

def pareto_threshold_calibration(
    proba: np.ndarray,
    y: np.ndarray,
    s: np.ndarray,
    acc_floor: float,
    grid_steps: int = 60,
) -> Tuple[float, float, float, float]:
    # Step 1: isotonic calibration per group
    proba_cal = proba.copy()
    for g in (0, 1):
        mask = s == g
        if mask.sum() < 10:
            continue
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(proba[mask], y[mask])
        proba_cal[mask] = iso.predict(proba[mask])

    # Step 2: grid search over (t0, t1)
    thresholds = np.linspace(0.15, 0.85, grid_steps)
    best_eo   = 1.0
    best_acc  = 0.0
    best_t0   = 0.5
    best_t1   = 0.5

    for t0 in thresholds:
        for t1 in thresholds:
            pred = np.where(s == 0, (proba_cal >= t0).astype(int),
                                    (proba_cal >= t1).astype(int))
            acc = accuracy_score(y, pred)
            if acc < acc_floor:
                continue
            try:
                eo = float(equalized_odds_difference(y, pred, sensitive_features=s))
            except Exception:
                continue
            if eo < best_eo or (eo == best_eo and acc > best_acc):
                best_eo  = eo
                best_acc = acc
                best_t0  = t0
                best_t1  = t1

    # Apply best thresholds
    pred_final = np.where(s == 0, (proba_cal >= best_t0).astype(int),
                                   (proba_cal >= best_t1).astype(int))
    final_acc = accuracy_score(y, pred_final)
    try:
        final_eo = float(equalized_odds_difference(y, pred_final, sensitive_features=s))
    except Exception:
        final_eo = best_eo

    return final_acc, final_eo, best_t0, best_t1


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 8 — Main Training: Single Unified Loop (no stage curriculum)
#
#  Why removed: the 3-stage curriculum was causing (a) cold-start instability
#  each stage reset, (b) PID integral bleed corrected but still suboptimal,
#  (c) Lawschool acc collapse in S1 from aggressive reweighing before any
#  fairness signal. Single loop with warmup schedule is simpler and more stable.
#
#  Structure:
#   epochs 1-warmup: task loss + mild adv only (build a good encoder first)
#   epochs warmup+1 to end: add EO loss + CF invariance + full adv
# ════════════════════════════════════════════════════════════════════════════

def _t(arr, dtype=torch.float32):
    return torch.tensor(np.asarray(arr), dtype=dtype).to(DEVICE)

def _proba(enc, hd, X, batch=2048):
    enc.eval(); hd.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            out.append(torch.sigmoid(hd(enc(X[i:i+batch]))).cpu().numpy())
    return np.concatenate(out)

def floor2(x):
    return math.floor(abs(float(x)) * 100) / 100

DATASET_ACC_FLOORS = {
    "adult": 0.78, "compas": 0.62, "german": 0.62,
    "bank": 0.77, "lawschool": 0.88, "hospital": 0.54,
}


def train_lcfr(
    dataset_name:       str,
    data:               Dict,
    causal_graphs:      List[CausalGraph],
    primary_sens_train: np.ndarray,
    primary_sens_test:  np.ndarray,
    baseline_acc:       float,
    total_epochs:       int = 300,
    warmup_epochs:      int = 60,
    batch_size:         int = 512,
    pid_target_eo:      float = 0.008,
) -> Tuple[float, float, float, float]:

    X_tr = _t(data['X_train_nn'])
    X_te = _t(data['X_test_nn'])
    y_tr = _t(data['y_train'],    torch.float32)
    y_te = _t(data['y_test'],     torch.float32)
    s_tr = _t(primary_sens_train, torch.long)
    s_te = _t(primary_sens_test,  torch.long)

    in_dim    = X_tr.shape[1]
    bbn_tr_df = data['bbn_train_df']
    bbn_cols  = list(bbn_tr_df.columns)

    primary_graph = causal_graphs[0] if causal_graphs else None
    cf_active = (primary_graph is not None and
                 primary_graph.sens_mi > CausalGraph.MI_THRESHOLD)
    mi_val = primary_graph.sens_mi if primary_graph else 0.0
    tqdm.write(f"    CF: {'ON' if cf_active else 'OFF'} (MI={mi_val:.4f})")
    # PRE: Causal-path-aware reweighing
    ck = CAUSAL_KNOWLEDGE.get(dataset_name, {})
    w_np = causal_reweigh(
        data['y_train'], primary_sens_train, bbn_tr_df,
        sens_col=ck.get("sens_cols", [""])[0],
        biased_mediators=[m for m in ck.get("biased_mediators", []) if m in bbn_tr_df.columns],
        effect_type=ck.get("effect_type", "direct"),
    )
    w_t = _t(w_np)

    if cf_active:
        cf_bbn_df = primary_graph.generate_counterfactuals(bbn_tr_df)
        cf_bbn_t  = _t(cf_bbn_df.values.astype(np.float32))
        bbn_tr_t  = _t(bbn_tr_df.values.astype(np.float32))
        dataset   = TensorDataset(X_tr, y_tr, s_tr, w_t, bbn_tr_t, cf_bbn_t)
    else:
        bbn_tr_t  = _t(bbn_tr_df.values.astype(np.float32))
        dataset   = TensorDataset(X_tr, y_tr, s_tr, w_t, bbn_tr_t)

    sampler = WeightedRandomSampler(w_np.tolist(), num_samples=len(w_np), replacement=True)
    loader  = DataLoader(dataset, batch_size=batch_size, sampler=sampler)

    enc     = FairnessEncoder(in_dim).to(DEVICE)
    task_hd = TaskHead().to(DEVICE)
    adv_hd  = AdversarialHead(z_dim=128).to(DEVICE)

    # CF projection: simple linear map from BBN feature space to X space
    # Replaces the over-parameterised CounterfactualRegularizer
    if cf_active:
        cf_proj = nn.Linear(len(bbn_cols), in_dim, bias=False).to(DEVICE)
    else:
        cf_proj = None

    pid = PIDController(target=pid_target_eo)
    bce = nn.BCEWithLogitsLoss(reduction='none')
    acc_floor = DATASET_ACC_FLOORS.get(dataset_name, baseline_acc * 0.90)

    all_params = list(enc.parameters()) + list(task_hd.parameters())
    if cf_proj is not None:
        all_params += list(cf_proj.parameters())

    opt_main = optim.AdamW(all_params, lr=2e-4, weight_decay=1e-4)
    opt_adv  = optim.AdamW(adv_hd.parameters(), lr=8e-4, weight_decay=1e-4)
    sched    = optim.lr_scheduler.CosineAnnealingLR(opt_main, T_max=total_epochs, eta_min=5e-6)

    best_state  = None
    best_metric = float('inf')

    # Track EO with a running mean (exact, not EMA — faster convergence signal)
    eo_history = []

    for ep in range(1, total_epochs + 1):
        enc.train(); task_hd.train(); adv_hd.train()
        in_fairness_phase = ep > warmup_epochs

        # GRL alpha schedule
        p     = max(0.0, (ep - warmup_epochs) / max(total_epochs - warmup_epochs, 1))
        alpha = 2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0

        adv_hd.set_alpha(alpha)

        ep_eo_vals = []

        for batch in loader:
            if cf_active:
                xb, yb, sb, wb, bbn_b, cf_b = batch
            else:
                xb, yb, sb, wb, bbn_b = batch
                cf_b = None

            z     = enc(xb)
            logit = task_hd(z)

            # Task loss
            loss = (bce(logit, yb) * wb).mean()

            # Adversarial (always on — mild during warmup via alpha=0 schedule)
            adv_logit = adv_hd(z)
            loss = loss + 0.5 * F.cross_entropy(adv_logit, sb)

            if in_fairness_phase:
                probs = torch.sigmoid(logit)

                # EO loss
                eo_val = eo_loss(logit, yb, sb, margin=0.005)
                w1_val = wasserstein_group_loss(probs, sb)

                with torch.no_grad():
                    eo_hard = float(torch.max(
                        torch.abs(probs[(sb==0)&(yb==1)].mean() - probs[(sb==1)&(yb==1)].mean()
                                  if ((sb==0)&(yb==1)).sum()>1 and ((sb==1)&(yb==1)).sum()>1
                                  else torch.tensor(0.0)),
                        torch.abs(probs[(sb==0)&(yb==0)].mean() - probs[(sb==1)&(yb==0)].mean()
                                  if ((sb==0)&(yb==0)).sum()>1 and ((sb==1)&(yb==0)).sum()>1
                                  else torch.tensor(0.0)),
                    ).item())
                    ep_eo_vals.append(eo_hard)

                # Linearly ramp EO weight over first 30 fairness epochs
                ramp = min(1.0, (ep - warmup_epochs) / 30.0)
                eo_w = pid.step(eo_hard) * ramp
                loss = loss + eo_w * (eo_val + 0.3 * w1_val)

                # CF invariance: z(x) ≈ z(x + cf_delta)
                if cf_active and cf_b is not None and cf_proj is not None:
                    delta = cf_proj(cf_b.float() - bbn_b.float())
                    z_cf  = enc(xb + delta)
                    cf_loss = 1.0 - F.cosine_similarity(z.detach(), z_cf, dim=-1).mean()
                    mi_scale = float(np.clip(primary_graph.sens_mi / 0.05, 0.5, 3.0))
                    loss = loss + mi_scale * cf_loss

            opt_main.zero_grad(); opt_adv.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
            nn.utils.clip_grad_norm_(task_hd.parameters(), 1.0)
            opt_main.step(); opt_adv.step()

        sched.step()

        if ep_eo_vals:
            eo_history.append(np.mean(ep_eo_vals))

        if ep % 60 == 0 or ep == total_epochs:
            enc.eval(); task_hd.eval()
            with torch.no_grad():
                pr_te = torch.sigmoid(task_hd(enc(X_te))).cpu().numpy()
            pred_te = (pr_te > 0.5).astype(int)
            acc = accuracy_score(y_te.cpu().numpy(), pred_te)
            try:
                eo = floor2(equalized_odds_difference(
                    y_te.cpu().numpy(), pred_te, sensitive_features=s_te.cpu().numpy()))
            except Exception:
                eo = 1.0

            tqdm.write(f"  ep{ep}: acc={acc:.4f} EO={eo:.4f} "
                       f"eo_w={pid.weight:.3f} phase={'fair' if in_fairness_phase else 'warm'}")

            acc_pen = max(0.0, acc_floor - acc) * 2.0
            metric  = eo + acc_pen
            if metric < best_metric:
                best_metric = metric
                best_state  = {
                    'enc':     copy.deepcopy(enc.state_dict()),
                    'task_hd': copy.deepcopy(task_hd.state_dict()),
                }

    if best_state:
        enc.load_state_dict(best_state['enc'])
        task_hd.load_state_dict(best_state['task_hd'])

    enc.eval(); task_hd.eval()
    proba_pre_post = _proba(enc, task_hd, X_te)
    pred_pre = (proba_pre_post > 0.5).astype(int)
    acc_pre  = accuracy_score(y_te.cpu().numpy(), pred_pre)
    try:
        eo_pre = floor2(equalized_odds_difference(
            y_te.cpu().numpy(), pred_pre, sensitive_features=s_te.cpu().numpy()))
    except Exception:
        eo_pre = 1.0

    # POST: Pareto-optimal group threshold calibration
    acc_post, eo_post, t0, t1 = pareto_threshold_calibration(
        proba_pre_post,
        y_te.cpu().numpy(),
        s_te.cpu().numpy(),
        acc_floor=acc_floor,
        grid_steps=60,
    )
    eo_post = floor2(eo_post)

    tqdm.write(f"\n  Pre-post : acc={acc_pre:.4f} EO={eo_pre:.4f}")
    tqdm.write(f"  Post-cal : acc={acc_post:.4f} EO={eo_post:.4f} (t0={t0:.3f}, t1={t1:.3f})")

    return acc_pre, eo_pre, acc_post, eo_post


# ════════════════════════════════════════════════════════════════════════════
#  SECTION 9 — Main Loop
# ════════════════════════════════════════════════════════════════════════════

print("=" * 65)
print("  LCFR v2 — Causal Reweighing + Unified Fairness Encoder")
print("  PRE:  Causal-path-aware instance reweighing (novel)")
print("  CORE: Single-loop FairnessEncoder + GRL + EO + W1 + CF invariance")
print("  POST: Pareto-optimal per-group isotonic threshold calibration (novel)")
print(f"  Device: {DEVICE}")
print("=" * 65)

DATASETS = {
    "adult":     preprocess_adult,
    "compas":    preprocess_compas,
    "german":    preprocess_german,
    "bank":      preprocess_bank,
    "lawschool": preprocess_lawschool,
    "hospital":  preprocess_hospital,
}

results = {}

for idx, (dname, loader_fn) in enumerate(DATASETS.items(), 1):
    print(f"\n{'─'*65}")
    print(f"  [{idx}/{len(DATASETS)}] {dname.upper()}")
    print(f"{'─'*65}")
    set_seed()

    data = loader_fn()

    for split in ('bbn_train_df', 'bbn_test_df'):
        df_fix = data[split]
        for col in df_fix.columns:
            if df_fix[col].dtype == object:
                data[split][col] = df_fix[col].astype('category').cat.codes.astype(int)

    sens_cfg = DATASET_CONFIG[dname]["sens_attrs"][0]
    s_tr_arr = np.asarray(data[sens_cfg[0]])
    s_te_arr = np.asarray(data[sens_cfg[1]])

    from sklearn.neural_network import MLPClassifier
    from sklearn.preprocessing import StandardScaler
    _sc  = StandardScaler()
    _Xb  = _sc.fit_transform(data['X_train_nn'])
    _Xbt = _sc.transform(data['X_test_nn'])
    _mlp = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=300, random_state=SEED)
    _mlp.fit(_Xb, data['y_train'])
    baseline_acc = accuracy_score(data['y_test'], _mlp.predict(_Xbt))
    tqdm.write(f"\n  {dname.upper()}  baseline_acc={baseline_acc:.4f}")

    tqdm.write(f"  Building causal graphs...")
    graphs = build_causal_graphs(dname, data['bbn_train_df'], data['y_train'])
    for g in graphs:
        tqdm.write(f"    sens={g.sens_col}  MI={g.sens_mi:.4f}  effect={g.effect_type}")

    acc_pre, eo_pre, acc_post, eo_post = train_lcfr(
        dataset_name=dname,
        data=data,
        causal_graphs=graphs,
        primary_sens_train=s_tr_arr,
        primary_sens_test=s_te_arr,
        baseline_acc=baseline_acc,
    )

    results[dname] = dict(
        baseline_acc=baseline_acc,
        acc_pre=acc_pre, eo_pre=eo_pre,
        acc_post=acc_post, eo_post=eo_post,
        drop_pre=baseline_acc - acc_pre,
        drop_post=baseline_acc - acc_post,
    )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'═'*65}")
print("  RESULTS SUMMARY")
print(f"{'═'*65}")
print(f"  {'Dataset':<12} {'Base':>7} | {'Acc(pre)':>9} {'EO(pre)':>8} | {'Acc(post)':>10} {'EO(post)':>9}")
print(f"  {'-'*60}")
for dname, r in results.items():
    print(f"  {dname:<12} {r['baseline_acc']:>7.4f} | "
          f"{r['acc_pre']:>9.4f} {r['eo_pre']:>8.4f} | "
          f"{r['acc_post']:>10.4f} {r['eo_post']:>9.4f}")
print(f"{'═'*65}")

  LCFR v2 — Causal Reweighing + Unified Fairness Encoder
  PRE:  Causal-path-aware instance reweighing (novel)
  CORE: Single-loop FairnessEncoder + GRL + EO + W1 + CF invariance
  POST: Pareto-optimal per-group isotonic threshold calibration (novel)
  Device: cuda

─────────────────────────────────────────────────────────────────
  [1/6] ADULT
─────────────────────────────────────────────────────────────────
  [cache] Adult

  ADULT  baseline_acc=0.8276
  Building causal graphs...
    sens=sex  MI=0.0260  effect=direct
    sens=race  MI=0.0061  effect=direct
    CF: ON (MI=0.0260)
  ep60: acc=0.7835 EO=0.1600 eo_w=1.000 phase=warm
  ep120: acc=0.7214 EO=0.0800 eo_w=2.171 phase=fair
  ep180: acc=0.7212 EO=0.0200 eo_w=2.730 phase=fair
  ep240: acc=0.7187 EO=0.0400 eo_w=2.447 phase=fair
  ep300: acc=0.7229 EO=0.0200 eo_w=2.391 phase=fair

  Pre-post : acc=0.7229 EO=0.0200
  Post-cal : acc=0.8157 EO=0.0200 (t0=0.589, t1=0.803)

─────────────────────────────────────────────────────────────